# ᚱ Viking Rune Stones: GPU Fine-Tuning Pipeline (Google Colab)
### Fine-Tuning YOLOv8 on Real Stone Textures & Inscriptions
---
**Overview:**
This notebook is specifically created for **Fine-Tuning** the pre-trained Viking Rune detector on real stone photos.
- **Hardware:** Accelerated by free Google Colab GPU (NVIDIA T4 / V100 / A100).
- **Speed:** 20-25 epochs take only ~2 minutes on T4 GPU!
- **Dataset:** Uses `viking_finetune_dataset.zip` (real rune stone photos & annotations).
- **Starting Weights:** Starts from `stage1_completed_best.pt` (or `best.pt`).


## 1. Verify GPU Accelerator & Install Dependencies


In [ ]:
!nvidia-smi

!pip install -q ultralytics pyyaml opencv-python pillow matplotlib seaborn

import os
import sys
import yaml
import zipfile
import shutil
from pathlib import Path
import torch
from ultralytics import YOLO

print(f"\n✓ PyTorch Version : {torch.__version__}")
print(f"✓ CUDA Available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✓ Active GPU      : {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ Warning: GPU is not enabled! Go to Runtime -> Change runtime type -> Select T4 GPU.")


## 2. Mount Google Drive (Optional) or Prepare Workspace
You can either upload files directly into Colab, or mount your Google Drive.


In [ ]:
# Option A: Mount Google Drive if your files are stored there
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("✓ Google Drive mounted!")
except Exception as e:
    print("ℹ️ Skipping Drive mount, using local Colab storage.")


## 3. Extract Fine-Tuning Dataset & Setup Configuration
Upload `viking_finetune_dataset.zip` and `stage1_completed_best.pt` (or `best.pt`) to Colab (drag and drop in the left file browser).


In [ ]:
# If the zip is in Google Drive, copy it here:
# !cp /content/drive/MyDrive/viking_finetune_dataset.zip .
# !cp /content/drive/MyDrive/stage1_completed_best.pt .

zip_path = "viking_finetune_dataset.zip"
if not os.path.exists(zip_path):
    raise FileNotFoundError("Please upload 'viking_finetune_dataset.zip' to the Colab files pane on the left!")

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(".")
print("✓ viking_finetune_dataset.zip extracted successfully!")

# Configure dataset.yaml paths for Google Colab
yaml_file = Path("viking_finetune_dataset/dataset.yaml")
with open(yaml_file, "r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

cfg["path"] = str(Path("viking_finetune_dataset").resolve())
with open(yaml_file, "w", encoding="utf-8") as f:
    yaml.dump(cfg, f, default_flow_style=False)

print(f"✓ dataset.yaml updated successfully! Root path: {cfg['path']}")
print(f"✓ Classes count: {cfg['nc']}")


## 4. Run GPU Fine-Tuning (Stage 2: Unfrozen End-to-End)
Fine-tunes all layers of the network with low learning rate (`lr0=0.0003`) preserving rune chirality (`fliplr=0.0`).


In [ ]:
# Detect starting weights (prefers stage1_completed_best.pt, falls back to best.pt)
starting_weights = None
for candidate in ["stage1_completed_best.pt", "best.pt", "viking_finetune_dataset/bestmodel/best.pt"]:
    if os.path.exists(candidate):
        starting_weights = candidate
        break

if not starting_weights:
    raise FileNotFoundError("Please upload 'stage1_completed_best.pt' or 'best.pt'!")

print(f"🚀 Starting Fine-Tuning from model: {starting_weights}")
model = YOLO(starting_weights)

# Run Fine-Tuning
results = model.train(
    data="viking_finetune_dataset/dataset.yaml",
    epochs=25,            # 25 epochs takes only ~2 minutes on T4 GPU!
    imgsz=640,
    batch=16,             # Colab T4 GPU has 16GB VRAM, batch=16 is fast and optimal
    device=0,             # GPU
    workers=4,
    freeze=0,             # Unfreeze all layers for end-to-end refinement
    lr0=0.0003,           # Low learning rate to protect pretrained features
    lrf=0.01,
    fliplr=0.0,           # CRITICAL: Runes are chiral! Never mirror horizontally
    flipud=0.0,
    degrees=8.0,          # Slight tilt variation
    scale=0.15,
    mosaic=0.3,
    close_mosaic=5,
    project="colab_finetune_runs",
    name="stage2_unfrozen_gpu",
    exist_ok=True,
    verbose=True
)
print("\n🏆 GPU Fine-Tuning Completed Successfully!")


## 5. Validate & Inspect Precision Metrics


In [ ]:
final_best = Path("colab_finetune_runs/stage2_unfrozen_gpu/weights/best.pt")
eval_model = YOLO(str(final_best))
val_metrics = eval_model.val(data="viking_finetune_dataset/dataset.yaml", imgsz=640, verbose=True)

print("=" * 60)
print(f"  Precision (P) : {val_metrics.box.mp:.4f} ({val_metrics.box.mp * 100:.2f}%)")
print(f"  Recall (R)    : {val_metrics.box.mr:.4f} ({val_metrics.box.mr * 100:.2f}%)")
print(f"  mAP@50        : {val_metrics.box.map50:.4f} ({val_metrics.box.map50 * 100:.2f}%)")
print(f"  mAP@50-95     : {val_metrics.box.map:.4f} ({val_metrics.box.map * 100:.2f}%)")
print("=" * 60)


## 6. Visualize Detections on Real Stone Images


In [ ]:
import glob
from PIL import Image
import matplotlib.pyplot as plt

test_images = glob.glob("viking_finetune_dataset/images/val/*.jpg")[:4]

plt.figure(figsize=(16, 12))
for idx, img_p in enumerate(test_images):
    res = eval_model.predict(img_p, conf=0.25)[0]
    annotated = res.plot()
    plt.subplot(2, 2, idx + 1)
    plt.imshow(annotated[..., ::-1])
    plt.title(f"Real Stone Validation: {Path(img_p).name}")
    plt.axis("off")

plt.tight_layout()
plt.show()


## 7. Download Final Master Model


In [ ]:
from google.colab import files

final_model_path = "viking_master_finetuned_best.pt"
shutil.copy2(str(final_best), final_model_path)
print(f"✓ Final model exported to: {final_model_path}")

# Download directly to your computer
files.download(final_model_path)
